# 🇺🇿 Notebook 2 — Lexicon + TF-IDF Baselines
**Project:** Uzbek Sentiment Analysis | **Author:** Asliddin

---
Two baselines before transformers:
1. **Lexicon** — rule-based, zero training, Uzbek word list
2. **TF-IDF + LR** — classical ML with character n-grams (better for Uzbek morphology)

**Expected:** Lexicon ~61%, TF-IDF ~73%

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from model import LexiconModel, BaselineModel
from evaluate import compute_metrics, plot_confusion_matrix
print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
LABEL2ID = {'negative':0,'neutral':1,'positive':2}
train_df = pd.read_csv('../data/processed/train.csv')
val_df   = pd.read_csv('../data/processed/val.csv')
test_df  = pd.read_csv('../data/processed/test.csv')

X_train, y_train = train_df['text'].tolist(), train_df['label'].map(LABEL2ID).tolist()
X_val,   y_val   = val_df['text'].tolist(),   val_df['label'].map(LABEL2ID).tolist()
X_test,  y_test  = test_df['text'].tolist(),  test_df['label'].map(LABEL2ID).tolist()

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

## 2. Lexicon Baseline (zero training)

In [ ]:
lexicon = LexiconModel()
lex_preds = lexicon.predict(X_test)
lex_metrics = compute_metrics(y_test, lex_preds)
print('Lexicon Baseline — Test Results:')
for k,v in lex_metrics.items(): print(f'  {k.capitalize():>10}: {v:.4f}')
print('\n', classification_report(y_test, lex_preds, target_names=['Negative','Neutral','Positive']))
plot_confusion_matrix(y_test, lex_preds, save_path='../results/lexicon_confusion.png')

## 3. TF-IDF + Logistic Regression

In [ ]:
baseline = BaselineModel(max_features=30000)
baseline.fit(X_train, y_train)

val_preds  = baseline.predict(X_val)
test_preds = baseline.predict(X_test)

print('TF-IDF Baseline — Validation:')
for k,v in compute_metrics(y_val, val_preds).items(): print(f'  {k.capitalize():>10}: {v:.4f}')
print('\nTF-IDF Baseline — Test:')
for k,v in compute_metrics(y_test, test_preds).items(): print(f'  {k.capitalize():>10}: {v:.4f}')
print('\n', classification_report(y_test, test_preds, target_names=['Negative','Neutral','Positive']))

baseline.save('../models/baseline.pkl')
plot_confusion_matrix(y_test, test_preds, save_path='../results/tfidf_confusion.png')

## 4. Baseline Comparison

In [ ]:
models   = ['Lexicon\n(zero-shot)', 'TF-IDF + LR\n(char n-gram)']
accs     = [compute_metrics(y_test,lex_preds)['accuracy'],
            compute_metrics(y_test,test_preds)['accuracy']]
f1s      = [compute_metrics(y_test,lex_preds)['f1'],
            compute_metrics(y_test,test_preds)['f1']]

fig, ax = plt.subplots(figsize=(8,5))
x = np.arange(len(models))
w = 0.32
b1 = ax.bar(x-w/2, accs, w, color=['#4fc3f7','#66bb6a'], alpha=0.9, label='Accuracy')
b2 = ax.bar(x+w/2, f1s,  w, color=['#4fc3f7','#66bb6a'], alpha=0.5, label='Macro F1')
for bar, val in zip(list(b1)+list(b2), accs+f1s):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_xticks(x); ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0.4, 0.9); ax.set_ylabel('Score')
ax.set_title('Baseline Comparison — Uzbek Sentiment', fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n→ Next: Notebook 03 — XLM-RoBERTa fine-tuning')